# Module 4 — Building an activity coefficient model from data

**2105603 Advanced Chemical Engineering Thermodynamics**
Department of Chemical Engineering, Chulalongkorn University
Soorathep Kheawhom

---

A published VLE table looks like a result. It is not. It is four measured
numbers per point — $T$, $P$, $x_1$, $y_1$ — and everything else in this
notebook is something you *derive* from them, under assumptions you choose.

This notebook takes one dataset the whole way:

| step | question |
|---|---|
| 1 | what was actually measured, and how well? |
| 2 | what is $\gamma_i$, and what did assuming it cost? |
| 3 | does the dataset obey Gibbs–Duhem? |
| 4 | which model, fitted how? |
| 5 | how uncertain are the parameters, really? |
| 6 | what did the fitting buy over predicting? |
| 7 | where may the answer be used? |

**Rule for this notebook.** Every number you report must be reproduced by an
independent route — a from-scratch calculation, a reference implementation, or
an analytical limit. A number produced once is a number you are trusting, not
a number you have checked.

## 0. Setup

Runs in Colab or on your own machine. `thermo` is optional; it is used only to
double-check the UNIFAC table against an independent implementation, which is
the kind of verification this course asks for.

In [ ]:
import sys, subprocess, importlib, os

def ensure(pkg, pipname=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        pipname or pkg], check=False)

for p in ("numpy", "scipy", "matplotlib"):
    ensure(p)
ensure("thermo")          # optional: only for cross-checking UNIFAC

# vlekit: use a local copy if there is one, otherwise fetch the course package
if not os.path.isdir("vlekit"):
    import urllib.request, zipfile, io
    URL = "https://www.skhgroup.net/teaching/2105603/files/vlekit.zip"
    try:
        with urllib.request.urlopen(URL, timeout=30) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
        print("downloaded vlekit from the course page")
    except Exception as e:
        print("could not download vlekit:", e)
        print("upload the vlekit folder to this session and re-run this cell")

import numpy as np
import matplotlib.pyplot as plt
import vlekit as v
from vlekit import consistency as C, plots as pl, unifac as U
from vlekit import flash as F, validate as VD

pl.use_style()
print("vlekit", v.__version__)

## 1. The dataset

Replace the block below with the table from your assigned paper. Keep the
citation in the `source` field — a dataset without a provenance is not usable
in a report.

If you have no dataset yet, leave `USE_SYNTHETIC = True`. The synthetic set is
generated from a known NRTL model, which has one large advantage for learning:
**you know the right answer**, so you can tell whether the pipeline works
before you point it at data whose answer nobody knows.

Note what the synthetic route costs you, though: data generated from NRTL will
of course be fitted best by NRTL. Any conclusion about *which model is best*
is meaningless on synthetic data. Only the machinery can be tested this way.

In [ ]:
USE_SYNTHETIC = True

c1 = v.component("2-propanol")     # component 1
c2 = v.component("mek")            # component 2
T_SET = 328.15                     # K, isothermal set

if USE_SYNTHETIC:
    TRUE = [0.35, 0.62]
    truth = v.NRTL(TRUE)
    dat = v.generate(truth, np.linspace(0.02, 0.98, 21), T=T_SET, c1=c1, c2=c2,
                     noise={"y": 0.003, "P": 0.05, "x": 0.001}, seed=3,
                     label="synthetic 2-propanol / MEK")
else:
    TRUE, truth = None, None
    # x1, y1, P/kPa   (for an isobaric set: x1, y1, T/K, and kind="isobaric")
    rows = [
        (0.050, 0.083, 43.9),
        (0.100, 0.152, 45.1),
        # ... paste the table here ...
    ]
    dat = v.from_table(rows, c1, c2, kind="isothermal", T=T_SET,
                       label="YOUR SYSTEM", source="Author, Journal, year, DOI")

# measurement uncertainties: change these to whatever the paper reports
dat.sigma = {"x": 0.001, "y": 0.002, "P": 0.05, "T": 0.05}

print(dat)
print(f"P from {dat.P.min():.2f} to {dat.P.max():.2f} kPa")
print(f"Psat1 = {c1.Psat(T_SET):.2f} kPa   Psat2 = {c2.Psat(T_SET):.2f} kPa")

### Check 1 — are the pure-component vapour pressures right?

Everything downstream divides by $P_i^{\rm sat}$. Before trusting a single
activity coefficient, verify the correlation at a condition you can look up:
the normal boiling point.

In [ ]:
for c, Tb_lit in ((c1, 355.40), (c2, 352.79)):     # literature normal bp / K
    Tb = float(c.Tsat(101.325))
    print(f"{c.name:12s} Antoine gives Tb = {Tb:7.2f} K, "
          f"literature {Tb_lit:7.2f} K, difference {Tb - Tb_lit:+.2f} K")
    if not np.all(c.in_range(T_SET)):
        print(f"   WARNING: {T_SET} K is outside the stated validity range "
              f"{c.Trange}")

## 2. From measurement to activity coefficient

$$\gamma_i = \frac{y_i \Phi_i P}{x_i P_i^{\rm sat}}, \qquad
\Phi_i = \frac{\hat\varphi_i}{\varphi_i^{\rm sat}}
\exp\!\left[-\frac{v_i^L (P - P_i^{\rm sat})}{RT}\right]$$

Setting $\Phi_i = 1$ is *modified Raoult's law*. It is an assumption about the
vapour, and you should know its size before you rely on it — so compute
$\gamma$ both ways and compare.

The uncertainty column is the one usually omitted. Note which term dominates.

In [ ]:
g1, g2 = dat.gamma("ideal")
s1, s2 = dat.gamma_sigma()

print(f"{'x1':>7}{'y1':>8}{'P/kPa':>9}{'gamma1':>10}{'gamma2':>10}"
      f"{'s(ln g1)':>11}{'s(ln g2)':>11}")
for i in range(0, dat.n, 2):
    print(f"{dat.x1[i]:>7.3f}{dat.y1[i]:>8.3f}{dat.P[i]:>9.2f}"
          f"{g1[i]:>10.4f}{g2[i]:>10.4f}{s1[i]:>11.4f}{s2[i]:>11.4f}")

**Question 2.1.** Which measured variable contributes most to
$\sigma(\ln\gamma_1)$ at $x_1 = 0.05$? At $x_1 = 0.5$? Change one $\sigma$ in
`dat.sigma` at a time and watch. Write down which instrument you would improve
first, and why that is not the same answer at both compositions.

In [ ]:
# how much does the vapour-phase assumption move the answer?
try:
    gv1, gv2 = dat.gamma("virial")
    d = np.abs(np.log(g1) - np.log(gv1))
    m = dat.interior() & np.isfinite(d)
    print(f"max |d ln gamma1| between ideal and virial: {np.max(d[m]):.5f}")
    print("(compare with the sigma column above: is the correction larger "
          "than the measurement uncertainty?)")
except Exception as e:
    print("virial route unavailable:", e)

## 3. Gibbs–Duhem: satisfied by construction, or by measurement?

$$x_1 \frac{{\rm d}\ln\gamma_1}{{\rm d}x_1}
    + x_2 \frac{{\rm d}\ln\gamma_2}{{\rm d}x_1} = 0
\qquad\text{(constant } T,P)$$

A model built from a single $G^E$ satisfies this **identically** — the residual
is zero to machine precision, and finding otherwise means the code is wrong.
Data satisfy it only to within their errors. The two statements look the same
on a slide and are completely different in practice.

In [ ]:
m = v.NRTL([0.4, 0.6])
err, ok = m.check_partial_molar()
print(f"model: ln gamma against numerical d(gE/RT)/dx1 -> max error {err:.2e} "
      f"({'OK' if ok else 'FAIL'})")
print(f"model: Gibbs-Duhem residual                    -> {m.gibbs_duhem_residual():.2e}")

xs, res, rms = C.gibbs_duhem_residual(dat)
print(f"data:  Gibbs-Duhem residual (RMS over the range) -> {rms:.3e}")

ax = pl.gibbs_duhem_plot(dat, model=m,
                         title="Gibbs-Duhem residual: data against model")
plt.show()

## 4. Thermodynamic consistency

Gibbs–Duhem makes a binary VLE dataset **over-determined**: one of $T$, $P$,
$x$, $y$ is redundant and can be predicted from the other three. The size of
the prediction error is the test. No model is assumed correct — only flexible
enough.

Five tests, and they are not equivalent:

| test | what it does | blind to |
|---|---|---|
| area (Redlich–Kister) | integrates $\ln(\gamma_1/\gamma_2)$ over the range | anything odd about $x_1 = \tfrac12$ |
| Herington $D-J$ | area test, with an empirical allowance for isobaric $\Delta T$ | same, plus it is a correlation not a derivation |
| point (Van Ness–Byer–Gibbs) | fits $P$–$x$ only, predicts $y$ | a model too rigid to follow $P$–$x$ |
| direct (Van Ness 1995) | residuals in $\ln(\gamma_1/\gamma_2)$, index 1–10 | same |
| infinite dilution (Kojima) | the two dilute ends | the middle |

In [ ]:
results = C.run_all(dat)
print(C.report(results))
for r in results:
    if r.note:
        print(f"\n{r.name}:\n  {r.note}")

In [ ]:
fit0 = v.barker_fit(dat, "nrtl")     # Barker: P-x only, y never fitted

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.4))
pl.area_plot(dat, ax=axes[0])
pl.point_test_plot(dat, fit0, ax=axes[1])
plt.tight_layout(); plt.show()

### The demonstration that matters

The area test integrates. Anything **odd** about $x_1 = \tfrac12$ integrates to
zero, so there is a whole family of errors it cannot see. This is not a
hypothetical: the cell below constructs one.

Perturbing $y_1$ shifts the measured ratio by
$\delta\ln(\gamma_1/\gamma_2) = \delta y_1 / [y_1(1-y_1)]$, so choosing
$\delta y_1 = \varepsilon\, y_1 (1-y_1)(2x_1 - 1)$ shifts it by
$\varepsilon(2x_1-1)$ — odd, and therefore invisible to the integral.

In [ ]:
from vlekit.regress import odd_bias

bad = v.generate(v.NRTL([0.35, 0.62]), np.linspace(0.02, 0.98, 21), T=T_SET,
                 c1=c1, c2=c2, noise={"y_bias": odd_bias(0.4)}, seed=1,
                 label="deliberately inconsistent")
print(C.report(C.run_all(bad)))

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.4))
pl.area_plot(bad, ax=axes[0])
pl.point_test_plot(bad, v.barker_fit(bad, "nrtl"), ax=axes[1])
plt.tight_layout(); plt.show()

**Question 4.1.** The area test passes this dataset with room to spare, and the
direct test gives it the worst possible index. Write two sentences you could
put in a referee report explaining why the authors' claim that "the data passed
the Redlich–Kister test" is not an answer.

**Question 4.2.** A dataset of yours fails the point test with residuals that
are all of one sign and roughly constant. Another fails with residuals that
change sign once, near the azeotrope. These have different likely causes. What
are they?

## 5. Regression — the objective function is a modelling decision

The same data and the same model give different parameters depending on what
you minimise. This is not numerical noise; it is a choice about which
measurement you trust.

- **`P` (Barker)** — uses $P$ and $x$ only. Standard, because $y$ is the least
  accurate measurement and this leaves it free to be predicted.
- **`y`** — fits the vapour composition. Tempting and usually wrong.
- **`gamma`** — fits $\ln\gamma$ directly. Weights the dilute ends heavily,
  because that is where $\ln\gamma$ is largest and least certain.
- **`Py`** — both, weighted by the stated $\sigma$.

In [ ]:
fits = v.compare_objectives(dat, "nrtl", objectives=("P", "y", "gamma", "Py"))
print(v.objective_table(fits, truth=TRUE))
if TRUE:
    print(f"generating values: tau12 = {TRUE[0]:+.4f}, tau21 = {TRUE[1]:+.4f}")

base = v.barker_fit(dat, "nrtl")
spread = np.ptp(np.array([f.params for f in fits]), axis=0)
print(f"\nspread across objectives : {spread}")
print(f"standard error from one   : {base.stderr}")

pl.objective_spread_plot(fits, truth=TRUE)
plt.show()

### Model comparison

A model with more parameters cannot fit worse. AIC penalises the extra
parameter, so it answers the question the residual cannot: *did the parameter
earn its place?*

In [ ]:
ms = v.compare_models(dat)
print(v.model_table(ms))

ax = pl.pxy(dat, ms[0], title=f"Best by AIC: {ms[0].model.name}")
plt.show()

**Question 5.1.** Look at the $\Delta$AIC column. How many of these models can
your dataset actually distinguish? A difference of less than about 2 is
conventionally regarded as no evidence at all. Write down the shortest honest
sentence describing which model the data support.

## 6. Uncertainty — the cloud, not the error bar

The standard errors printed by a least-squares routine come from linearising
around the optimum and assuming normal residuals. For two parameters and twenty
points, neither assumption holds well. Resampling shows the real shape.

In [ ]:
boot = v.bootstrap(dat, "nrtl", n_boot=300, seed=2)
lo, hi = boot["ci95"]
for i, n in enumerate(base.model.param_names):
    line = (f"  {n:<8}{base.params[i]:+.4f}   "
            f"linearised +/- {base.stderr[i]:.4f}   "
            f"bootstrap 95% [{lo[i]:+.4f}, {hi[i]:+.4f}]")
    if TRUE:
        line += f"   true {TRUE[i]:+.4f}"
    print(line)
print(f"\n  parameter correlation: {boot['corr'][0, 1]:+.3f}")

pl.bootstrap_plot(boot, truth=TRUE)
plt.show()

**Question 6.1.** With a correlation of this magnitude, what does quoting
$\tau_{12} \pm \sigma_{12}$ and $\tau_{21} \pm \sigma_{21}$ separately claim
that the data do not support? Sketch the region those two intervals describe
and compare it with the cloud.

## 7. What did the fitting buy?

UNIFAC predicts $\gamma$ from group contributions with **nothing fitted to your
data**. A regressed model must beat it on the data it was fitted to — that is
not an achievement, it saw the answer. The question is by how much, and whether
the margin survives outside the fitted range.

On synthetic data this comparison is rigged, because the "truth" is the same
functional form as the fitted model. Interpret it only on real data.

In [ ]:
rows = U.benchmark(dat, [v.barker_fit(dat, "nrtl"), v.barker_fit(dat, "uniquac")])
print(U.benchmark_table(rows))

xs = np.linspace(1e-4, 1 - 1e-4, 200)
uf1, uf2 = U.gammas(xs, T_SET, c1, c2)
ax = pl.gamma_plot(dat, base, title="Fitted NRTL against unfitted UNIFAC")
ax.plot(xs, uf1, ":", color=pl.RUST, lw=2.2, label="UNIFAC $\\gamma_1$")
ax.plot(xs, uf2, ":", color=pl.NAVY, lw=2.2, label="UNIFAC $\\gamma_2$")
ax.legend(loc="best", fontsize=9.5)
plt.show()

### Check 2 — is the UNIFAC implementation right?

`vlekit` carries its own group-contribution code so you can read it. That is
only defensible if it is verified against something independent.

In [ ]:
try:
    from thermo.unifac import UNIFAC_gammas
    worst = 0.0
    for x in (0.1, 0.3, 0.5, 0.7, 0.9):
        mine = U.gammas(np.array([x]), T_SET, c1, c2)
        ref = UNIFAC_gammas(T=T_SET, xs=[x, 1 - x],
                            chemgroups=[c1.groups, c2.groups])
        worst = max(worst, abs(mine[0][0] - ref[0]), abs(mine[1][0] - ref[1]))
    print(f"max difference against the `thermo` implementation: {worst:.2e}")
except ImportError:
    print("thermo not installed - the independent check did not run")

## 8. Has the model learned the system, or the noise?

Reporting the residual on the points you fitted is not evidence. Hold out part
of the composition range and predict it.

The `middle` split — train on the centre, predict the dilute ends — is the
useful one, because the ends are where models differ and where the numbers a
designer needs actually live.

In [ ]:
cvs = []
for how in ("middle", "ends", "even"):
    r = VD.cross_validate(dat, "nrtl", how=how)
    r["how"] = how
    cvs.append(r)
print(VD.cross_validation_table(cvs))

pl.cross_validation_plot(cvs[0])
plt.show()

**Question 8.1.** The `even` split — train on every other point — usually looks
excellent. Explain in one sentence why it tests almost nothing.

## 9. What the parameters claim about regions you did not measure

Two parameter sets that agree over the measured range can disagree completely
outside it: about $\gamma^\infty$, about whether there is an azeotrope, and
about whether the liquid splits into two phases. That is not a defect of the
fitting. It is what it means for data to be silent about a region.

The cell below fits a dataset that covers only part of the composition range —
the common situation with older papers — and asks each model what it implies.

In [ ]:
hx, me = v.component("n-hexane"), v.component("methanol")
partial = v.generate(v.Margules([2.4, 2.0]), np.linspace(0.55, 0.98, 14),
                     T=318.15, c1=hx, c2=me,
                     noise={"y": 0.003, "P": 0.08}, seed=5,
                     label="measured only for x1 > 0.55")

pfits = [v.fit(partial, m) for m in ("margules", "nrtl", "uniquac", "wilson")]
named = [(f"{f.model.name} (rms dy {f.rms_y:.5f})", f.model) for f in pfits]
print(VD.implications_table(named, 318.15, hx, me))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.6))
pl.extrapolation_plot(partial, pfits[:3], ax=axes[0])
pl.gmix_plot([f.model for f in pfits[:3]], T=318.15,
             labels=[f.model.name for f in pfits[:3]],
             title="Gibbs energy of mixing, and the common tangent", ax=axes[1])
plt.tight_layout(); plt.show()

**Question 9.1.** All four models reproduce the measurements to within the
scatter. They report $\gamma_1^\infty$ values differing by tens of per cent,
and they do not agree on where the liquid splits. If you had to size a stripper
that operates at $x_1 = 0.02$, which number would you use, and what extra
measurement would you ask for first?

**Question 9.2.** Wilson refuses to predict a split for any parameters — it is
structurally incapable of it. Look at what it does to $\gamma_1^\infty$
instead. Why is that number a symptom rather than a result?

## 10. Using the model: bubble, dew, flash

The Rachford–Rice function is monotonic in the vapour fraction, which is why it
is the formulation everyone uses: a monotonic function has one root and
bisection cannot fail. Worth seeing once rather than being told.

In [ ]:
z1, Tf = 0.45, T_SET
Pb, _ = v.bubble_P(base.model, np.array([z1]), np.array([Tf]), c1, c2)
Pd, _ = F.dew_P(base.model, np.array([z1]), np.array([Tf]), c1, c2)
Pf = 0.5 * (float(Pb[0]) + float(Pd[0]))
print(f"at z1 = {z1}: bubble P = {float(Pb[0]):.2f} kPa, "
      f"dew P = {float(Pd[0]):.2f} kPa; flashing at {Pf:.2f} kPa")

Vf, xf, yf, status = F.flash_TP(base.model, z1, Tf, Pf, c1, c2)
print(f"  V = {Vf:.4f}, x1 = {xf:.4f}, y1 = {yf:.4f}  ({status})")
print(f"  mass balance residual: {Vf * yf + (1 - Vf) * xf - z1:+.2e}")

K1, K2 = F.K_values(base.model, np.array([xf]), np.array([Tf]), Pf, c1, c2)
pl.rachford_rice_plot(z1, float(K1[0]), float(K2[0]))
plt.show()

az = F.azeotrope(base.model, Tf, c1, c2)
print("\nazeotrope:", az)

## 11. The deliverable

**A regression does not deliver a parameter set. It delivers a parameter set
plus a statement of where it may be used.** Anything less is not a result, and
in this course it is not a pass.

The cell below drafts that statement from what was actually done. Read it
critically — it is a draft, not a substitute for your judgement — and edit it
into your report.

In [ ]:
print(VD.domain_statement(base, dat, unifac_rows=rows))

---

## Exercises

**E1. Your dataset.** Set `USE_SYNTHETIC = False`, paste in the table from your
assigned paper, and run the notebook end to end. Report: the consistency test
table, the model you chose with your reason, the bootstrap confidence region,
and the domain of validity statement.

**E2. The vapour-phase assumption.** Run your dataset with `vapour="ideal"` and
`vapour="virial"` throughout. Does any consistency verdict change? If it does,
say which assumption the data were actually testing.

**E3. Diagnosis, not verdict.** Take your dataset and corrupt it three ways:
(a) a constant offset in $y$, (b) a $0.3$ K error in $T$, (c) Antoine constants
for the wrong isomer of one component. Run the five tests each time. Tabulate
the *signature* of each fault — which tests fail, and what shape the point-test
residual takes. This table is the useful part of consistency testing.

**E4. Cross-validation done properly.** Fit your dataset on $x_1 < 0.5$ and
predict $x_1 > 0.5$, then the reverse. Report both. If the two disagree, say
what that tells you about the model rather than about the data.

**E5. Against a prediction.** Compare your best fit with UNIFAC over the full
composition range, not just where the data are. State the margin in a quantity
someone would actually use — $\gamma^\infty$, or the azeotrope composition —
rather than in rms.

---

## AI checkpoint

Language models are permitted and expected here. They are also very good at
producing plausible regressions that the data do not support, so the exercise
is to test that rather than to trust it.

1. Ask a language model to propose an objective function and starting values
   for your system, and to justify the choice. Paste its answer into your
   report verbatim.
2. Implement its proposal in this notebook.
3. Test it: does the resulting fit survive the `middle` cross-validation split?
   Is its bootstrap cloud narrower or wider than Barker's? Does it change any
   consistency verdict?
4. Write two paragraphs: **did the model help, or did it produce a plausible
   answer the data do not support?** Cite the specific numbers that decided it.

An answer that says "the model was helpful" without a number that could have
come out the other way scores zero. So does an unverified machine-produced
result, whether or not it happens to be right.

---

*vlekit and this notebook: Soorathep Kheawhom, 2105603, Chulalongkorn
University.*